# 05d — Utility-Preserving Dynamic EO Adversarial Fine-Tuning

This notebook supersedes the current experimental `05c` implementation without overwriting it.

## Main corrections

1. **Twelve-epoch maximum search window.** This matches the maximum epoch count used by the baseline and original static/dynamic notebooks, while transparently retaining the extra baseline-initialisation cost of a two-stage method.
2. **One common checkpoint epoch across all seeds.** The epoch is selected from aggregate validation performance, rather than forcing each seed to satisfy a noisy seed-specific EO target.
3. **No artificial “wait until the ramp ends” rule.** Any epoch exposed to the dynamic adversarial schedule can be selected.
4. **Ramped projection strength.** A tiny adversarial coefficient no longer activates a full gradient projection.
5. **Gradient-norm balancing.** The reversed fairness gradient is scaled relative to the disease-gradient norm, preventing raw adversary-gradient magnitude from dominating.
6. **EO-cell balancing.** Each per-label adversary balances the four `(true label, sex)` cells, so rare positive strata are not overwhelmed by negatives.
7. **Utility-preserving fine-tuning.** The disease head is adapted first; `layer4` is gradually unfrozen at a very low learning rate, and all ResNet BatchNorm statistics remain fixed.

## Scientific interpretation

This is an enhanced dynamic adversarial method, not evidence that dynamic scheduling alone is superior. For a publication-quality schedule ablation, duplicate the notebook and change only `SCHEDULE_MODE` from `"dynamic"` to `"static_matched"`.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torchvision import models

from r50_multilabel_final_common import (
    aggregate_multi_seed_table,
    build_label_fairness_summary,
    build_loaders,
    compute_test_metrics_by_group,
    ensure_output_dirs,
    get_device,
    load_fixed_multilabel_data,
    make_disease_criterion,
    patient_cluster_bootstrap_multiseed,
    per_label_ranking_metrics,
    safe_auroc,
    save_prediction_archive,
    seed_everything,
    select_per_label_thresholds,
    summarise_overall_fairness,
)

# =============================================================================
# 1. Pre-specified experimental protocol
# =============================================================================
FINAL_SEEDS = [42, 123, 2026]
BATCH_SIZE = 16

# Use the same 12-epoch maximum checkpoint-search window as notebooks 03/04/05.
# Because this method starts from a completed baseline checkpoint, this does not
# imply equal total compute; that distinction must be reported transparently.
MAX_FINE_TUNE_EPOCHS = 12

# The baseline checkpoint is already task-trained, so Stage 2 only strengthens
# the conditional adversary before the predictor is allowed to move.
ADVERSARY_WARMUP_EPOCHS = 2

# Dynamic GRL / reversed-gradient schedule.
SCHEDULE_MODE = "dynamic"  # Set only to "static_matched" for the matched ablation.
LAMBDA_MAX = 0.10
RAMP_END_EPOCH = 3.0

# Utility-preserving partial fine-tuning.
HEAD_ONLY_EPOCHS = 1
LAYER4_LR = 2e-6
DISEASE_HEAD_LR = 1e-5
ADVERSARY_LR = 3e-4
WEIGHT_DECAY = 1e-4

# Conditional cell reweighting balances (true label, sex) strata per disease.
EO_CELL_WEIGHT_CLIP = 20.0

# One common epoch is selected across seeds using aggregate validation results.
STATIC_EO_TOLERANCE = 0.002
THRESHOLD_METRIC = "f1"

EXPERIMENT_ID = (
    "r50_dynamic_eo_gradnorm_common_epoch_v2"
    if SCHEDULE_MODE == "dynamic"
    else "r50_static_matched_eo_gradnorm_common_epoch_v2"
)

RUN_PATIENT_CLUSTER_BOOTSTRAP = False
N_BOOTSTRAP = 1000
BOOTSTRAP_RANDOM_SEED = 202606
DELETE_NONSELECTED_EPOCH_CHECKPOINTS = True

ensure_output_dirs()
Path("figures").mkdir(exist_ok=True)
device = get_device()

print("Device:", device)
print("Experiment:", EXPERIMENT_ID)
print("Schedule mode:", SCHEDULE_MODE)
print("Seeds:", FINAL_SEEDS)
print("Maximum fine-tuning epochs:", MAX_FINE_TUNE_EPOCHS)

Device: mps
Experiment: r50_dynamic_eo_gradnorm_common_epoch_v2
Schedule mode: dynamic
Seeds: [42, 123, 2026]
Maximum fine-tuning epochs: 12


## Fixed patient-level dataset and seven-label protocol

In [2]:
train_df, val_df, test_df, selected_labels, label_columns, audit_config = (
    load_fixed_multilabel_data()
)

train_patients = set(train_df["Patient ID"].astype(str))
val_patients = set(val_df["Patient ID"].astype(str))
test_patients = set(test_df["Patient ID"].astype(str))
assert train_patients.isdisjoint(val_patients)
assert train_patients.isdisjoint(test_patients)
assert val_patients.isdisjoint(test_patients)

criterion_for_weights, pos_weight_table, pos_weight_values = make_disease_criterion(
    train_df, label_columns, device
)
del criterion_for_weights

print("Selected labels:", selected_labels)
print(
    "Split sizes:",
    {"train": len(train_df), "validation": len(val_df), "test": len(test_df)},
)
display(pos_weight_table)

Selected labels: ['Infiltration', 'Effusion', 'Atelectasis', 'Nodule', 'Mass', 'Pneumothorax', 'Consolidation']
Split sizes: {'train': 78873, 'validation': 10953, 'test': 22294}


,label,train_positive,train_negative,pos_weight
0,Infiltration,13868,65005,4.687410
1,Effusion,9533,69340,7.273681
2,Atelectasis,8262,70611,8.546478
3,Nodule,4415,74458,16.864779
4,Mass,4199,74674,17.783758
5,Pneumothorax,3841,75032,19.534496
6,Consolidation,3280,75593,23.046646


## EO-conditioned model

In [3]:
class PerLabelEOAdversary(nn.Module):
    """One conditional sex adversary per disease label.

    Head k receives [sigmoid(disease_logit_k), true_label_k]. Predicting sex from
    the model output conditional on the true label is the adversarial proxy for
    equalized odds.
    """

    def __init__(self, n_labels: int, hidden_dim: int = 32):
        super().__init__()
        self.n_labels = int(n_labels)
        self.heads = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(2, hidden_dim),
                    nn.ReLU(inplace=True),
                    nn.Linear(hidden_dim, hidden_dim),
                    nn.ReLU(inplace=True),
                    nn.Linear(hidden_dim, 1),
                )
                for _ in range(self.n_labels)
            ]
        )

    def forward(
        self,
        disease_logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        disease_probs = torch.sigmoid(disease_logits)
        outputs = []
        for label_index, head in enumerate(self.heads):
            conditional_input = torch.stack(
                [
                    disease_probs[:, label_index],
                    targets[:, label_index],
                ],
                dim=1,
            )
            outputs.append(head(conditional_input))
        return torch.cat(outputs, dim=1)


class DynamicEOProjectedResNet50(nn.Module):
    """ResNet-50 disease predictor with per-label EO adversaries."""

    def __init__(self, n_labels: int):
        super().__init__()
        self.backbone = models.resnet50(weights=None)
        self.feature_dim = self.backbone.fc.in_features
        if self.feature_dim != 2048:
            raise RuntimeError(
                f"Unexpected ResNet-50 feature dimension: {self.feature_dim}"
            )
        self.backbone.fc = nn.Identity()
        self.disease_head = nn.Linear(self.feature_dim, n_labels)
        self.eo_adversary = PerLabelEOAdversary(n_labels=n_labels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.disease_head(features)

    def conditional_sex_logits(
        self,
        disease_logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        return self.eo_adversary(disease_logits, targets)


def safe_torch_load(path: Path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _normalise_checkpoint_keys(state_dict):
    if state_dict and all(key.startswith("module.") for key in state_dict):
        return {
            key.replace("module.", "", 1): value
            for key, value in state_dict.items()
        }
    return state_dict


def load_baseline_checkpoint_into_dynamic_model(
    model: DynamicEOProjectedResNet50,
    checkpoint_path: Path,
    n_labels: int,
):
    """Map notebook-03 baseline weights into the expanded dynamic model."""
    payload = safe_torch_load(checkpoint_path, map_location="cpu")
    state_dict = _normalise_checkpoint_keys(payload["model_state_dict"])

    baseline = models.resnet50(weights=None)
    baseline_feature_dim = baseline.fc.in_features
    baseline.fc = nn.Linear(baseline_feature_dim, n_labels)
    missing, unexpected = baseline.load_state_dict(state_dict, strict=False)

    if missing or unexpected:
        raise RuntimeError(
            "Baseline checkpoint did not map cleanly. "
            f"Missing={missing}; unexpected={unexpected}"
        )

    backbone_state = {
        key: value
        for key, value in baseline.state_dict().items()
        if not key.startswith("fc.")
    }
    missing_backbone, unexpected_backbone = model.backbone.load_state_dict(
        backbone_state,
        strict=False,
    )
    allowed_missing = {"fc.weight", "fc.bias"}
    if set(missing_backbone) - allowed_missing or unexpected_backbone:
        raise RuntimeError(
            "Backbone checkpoint did not map cleanly. "
            f"Missing={missing_backbone}; unexpected={unexpected_backbone}"
        )

    model.disease_head.load_state_dict(baseline.fc.state_dict())
    return payload

## Balance true-label/sex cells for each disease

In [4]:
def build_eo_cell_weights(
    dataframe: pd.DataFrame,
    label_columns: Sequence[str],
    clip_max: float,
) -> Tuple[torch.Tensor, pd.DataFrame]:
    """Create inverse-frequency weights for each (label, y, sex) cell.

    For each disease label, the four conditional cells y in {0,1}, sex in {0,1}
    receive equal total weight before clipping. This prevents rare positive
    strata from being overwhelmed by the much larger negative strata.
    """
    n = len(dataframe)
    table = np.zeros((len(label_columns), 2, 2), dtype=np.float32)
    rows = []

    for label_index, column in enumerate(label_columns):
        y = dataframe[column].to_numpy(dtype=int)
        s = dataframe["sex"].to_numpy(dtype=int)

        for y_value in (0, 1):
            for s_value in (0, 1):
                count = int(np.sum((y == y_value) & (s == s_value)))
                if count <= 0:
                    raise ValueError(
                        f"Empty EO cell for {column}, y={y_value}, sex={s_value}."
                    )
                raw_weight = n / (4.0 * count)
                clipped_weight = float(min(raw_weight, clip_max))
                table[label_index, y_value, s_value] = clipped_weight
                rows.append(
                    {
                        "label": column.replace("label_", "", 1),
                        "target": y_value,
                        "sex": s_value,
                        "count": count,
                        "raw_inverse_frequency_weight": raw_weight,
                        "used_weight": clipped_weight,
                    }
                )

        # Normalise each disease label to mean sample weight 1 on the train set.
        sample_weight_sum = 0.0
        for y_value in (0, 1):
            for s_value in (0, 1):
                count = np.sum((y == y_value) & (s == s_value))
                sample_weight_sum += count * table[label_index, y_value, s_value]
        normaliser = n / sample_weight_sum
        table[label_index] *= normaliser

    weight_df = pd.DataFrame(rows)
    return torch.tensor(table, dtype=torch.float32), weight_df


eo_cell_weights_cpu, eo_cell_weight_table = build_eo_cell_weights(
    train_df,
    label_columns,
    clip_max=EO_CELL_WEIGHT_CLIP,
)
display(eo_cell_weight_table)


def conditional_adv_loss(
    adv_logits: torch.Tensor,
    targets: torch.Tensor,
    sexes: torch.Tensor,
    eo_cell_weights: torch.Tensor,
) -> torch.Tensor:
    """Balanced per-label BCE over true-label/sex conditional cells."""
    sexes_long = sexes.long()
    targets_long = targets.long()
    sex_targets = sexes.float().unsqueeze(1).expand_as(adv_logits)

    elementwise_bce = F.binary_cross_entropy_with_logits(
        adv_logits,
        sex_targets,
        reduction="none",
    )

    batch_weights = []
    for label_index in range(adv_logits.shape[1]):
        batch_weights.append(
            eo_cell_weights[
                label_index,
                targets_long[:, label_index],
                sexes_long,
            ]
        )
    batch_weights = torch.stack(batch_weights, dim=1)

    return (elementwise_bce * batch_weights).sum() / batch_weights.sum().clamp_min(1e-12)

,label,target,sex,count,raw_inverse_frequency_weight,used_weight
0,Infiltration,0,0,28511,0.691601,0.691601
1,Infiltration,0,1,36494,0.540315,0.540315
2,Infiltration,1,0,5925,3.327975,3.327975
3,Infiltration,1,1,7943,2.482469,2.482469
4,Effusion,0,0,30216,0.652576,0.652576
5,Effusion,0,1,39124,0.503994,0.503994
6,Effusion,1,0,4220,4.672571,4.672571
7,Effusion,1,1,5313,3.711321,3.711321
8,Atelectasis,0,0,31073,0.634578,0.634578
9,Atelectasis,0,1,39538,0.498716,0.498716


## Utility-preserving dynamic projected optimisation

In [5]:
def freeze_batch_norm(module: nn.Module) -> None:
    for child in module.modules():
        if isinstance(child, nn.BatchNorm2d):
            child.eval()
            for parameter in child.parameters():
                parameter.requires_grad = False


def set_adversary_warmup_trainability(model: DynamicEOProjectedResNet50) -> None:
    for parameter in model.backbone.parameters():
        parameter.requires_grad = False
    for parameter in model.disease_head.parameters():
        parameter.requires_grad = False
    for parameter in model.eo_adversary.parameters():
        parameter.requires_grad = True

    model.eval()
    model.eo_adversary.train()


def set_predictor_trainability(
    model: DynamicEOProjectedResNet50,
    fine_tune_epoch: int,
) -> None:
    for parameter in model.backbone.parameters():
        parameter.requires_grad = False

    # Gradual unfreezing: start with the disease head, then allow layer4
    # convolutions to move at a much smaller learning rate.
    if fine_tune_epoch > HEAD_ONLY_EPOCHS:
        for parameter in model.backbone.layer4.parameters():
            parameter.requires_grad = True
        freeze_batch_norm(model.backbone.layer4)

    for parameter in model.disease_head.parameters():
        parameter.requires_grad = True
    for parameter in model.eo_adversary.parameters():
        parameter.requires_grad = True


def set_predictor_train_mode(model: DynamicEOProjectedResNet50) -> None:
    # Keep all ResNet BatchNorm running statistics fixed. This is important for
    # small-batch fine-tuning from an already selected diagnostic checkpoint.
    model.backbone.eval()
    model.disease_head.train()
    model.eo_adversary.train()


def lambda_from_step(global_step: int, steps_per_epoch: int) -> float:
    if SCHEDULE_MODE == "static_matched":
        return float(LAMBDA_MAX)
    if SCHEDULE_MODE != "dynamic":
        raise ValueError(f"Unsupported SCHEDULE_MODE={SCHEDULE_MODE!r}")

    epoch_position = global_step / max(steps_per_epoch, 1)
    if epoch_position >= RAMP_END_EPOCH:
        return float(LAMBDA_MAX)

    phase = float(np.clip(epoch_position / RAMP_END_EPOCH, 0.0, 1.0))
    return float(LAMBDA_MAX * 0.5 * (1.0 - math.cos(math.pi * phase)))


def train_adversary_warmup_epoch(
    model,
    loader,
    optimizer_adversary,
    eo_cell_weights,
    device,
):
    set_adversary_warmup_trainability(model)
    total_loss = 0.0

    for images, targets, sexes, _, _ in loader:
        images = images.to(device)
        targets = targets.to(device)
        sexes = sexes.to(device)

        optimizer_adversary.zero_grad(set_to_none=True)
        with torch.no_grad():
            disease_logits = model(images)
        adv_logits = model.conditional_sex_logits(disease_logits.detach(), targets)
        loss = conditional_adv_loss(
            adv_logits,
            targets,
            sexes,
            eo_cell_weights,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.eo_adversary.parameters(), max_norm=5.0)
        optimizer_adversary.step()
        total_loss += loss.item() * images.size(0)

    return float(total_loss / len(loader.dataset))


def train_gradnorm_projected_epoch(
    model,
    loader,
    optimizer_predictor,
    optimizer_adversary,
    disease_criterion,
    eo_cell_weights,
    device,
    fine_tune_epoch,
    global_step,
):
    """Utility-preserving reversed-gradient update.

    Key differences from the original 05c:
    1. Projection strength ramps with lambda instead of switching on fully.
    2. The reversed adversarial gradient is norm-balanced to a lambda fraction
       of the disease-gradient norm.
    3. BatchNorm statistics remain frozen.
    """
    set_predictor_trainability(model, fine_tune_epoch)

    shared_parameters = [
        parameter
        for parameter in (
            list(model.backbone.layer4.parameters())
            + list(model.disease_head.parameters())
        )
        if parameter.requires_grad
    ]
    adversary_parameters = list(model.eo_adversary.parameters())

    total_disease_loss = 0.0
    total_adv_loss = 0.0
    lambda_values = []
    cosine_values = []
    task_norm_values = []
    adv_norm_values = []
    effective_adv_ratio_values = []
    conflict_flags = []
    schedule_rows = []

    for batch_index, (images, targets, sexes, _, _) in enumerate(loader, start=1):
        set_predictor_train_mode(model)

        images = images.to(device)
        targets = targets.to(device)
        sexes = sexes.to(device)
        lambda_current = lambda_from_step(
            global_step=global_step,
            steps_per_epoch=len(loader),
        )

        # ---------------- Predictor update ----------------
        optimizer_predictor.zero_grad(set_to_none=True)
        for parameter in adversary_parameters:
            parameter.requires_grad = False

        disease_logits = model(images)
        adv_logits = model.conditional_sex_logits(disease_logits, targets)
        disease_loss = disease_criterion(disease_logits, targets)
        adv_loss_for_predictor = conditional_adv_loss(
            adv_logits,
            targets,
            sexes,
            eo_cell_weights,
        )

        task_grads = torch.autograd.grad(
            disease_loss,
            shared_parameters,
            retain_graph=True,
            allow_unused=True,
        )
        adv_grads = torch.autograd.grad(
            adv_loss_for_predictor,
            shared_parameters,
            retain_graph=False,
            allow_unused=True,
        )

        task_grads = [
            gradient if gradient is not None else torch.zeros_like(parameter)
            for gradient, parameter in zip(task_grads, shared_parameters)
        ]
        adv_grads = [
            gradient if gradient is not None else torch.zeros_like(parameter)
            for gradient, parameter in zip(adv_grads, shared_parameters)
        ]

        dot_product = sum(
            (task * adv).sum()
            for task, adv in zip(task_grads, adv_grads)
        )
        task_norm_sq = sum((task * task).sum() for task in task_grads)
        adv_norm_sq = sum((adv * adv).sum() for adv in adv_grads)
        task_norm = torch.sqrt(task_norm_sq + 1e-12)
        adv_norm = torch.sqrt(adv_norm_sq + 1e-12)

        conflict = bool(dot_product.item() > 0.0)
        lambda_fraction = (
            float(lambda_current / LAMBDA_MAX)
            if LAMBDA_MAX > 0.0
            else 0.0
        )

        # Ramped PCGrad-style removal of the task component that helps the
        # adversary. Unlike the original 05c, tiny lambda no longer activates a
        # full projection discontinuously.
        if conflict and adv_norm_sq.item() > 0.0:
            full_projection_scale = dot_product / (adv_norm_sq + 1e-12)
            projection_scale = lambda_fraction * full_projection_scale
        else:
            projection_scale = torch.tensor(0.0, device=device)

        # Norm-balanced reversed gradient. At lambda=0.1, the fairness component
        # has approximately 10% of the task-gradient norm, independent of the raw
        # adversary-gradient scale.
        reverse_scale = (
            lambda_current * task_norm / (adv_norm + 1e-12)
            if adv_norm_sq.item() > 0.0
            else torch.tensor(0.0, device=device)
        )

        for parameter, task_gradient, adv_gradient in zip(
            shared_parameters,
            task_grads,
            adv_grads,
        ):
            projected_task_gradient = (
                task_gradient - projection_scale * adv_gradient
            )
            parameter.grad = (
                projected_task_gradient - reverse_scale * adv_gradient
            )

        torch.nn.utils.clip_grad_norm_(shared_parameters, max_norm=5.0)
        optimizer_predictor.step()

        cosine = (
            dot_product / (task_norm * adv_norm + 1e-12)
            if task_norm_sq.item() > 0.0 and adv_norm_sq.item() > 0.0
            else torch.tensor(np.nan, device=device)
        )

        # ---------------- Adversary update ----------------
        for parameter in adversary_parameters:
            parameter.requires_grad = True

        model.eval()
        model.eo_adversary.train()
        optimizer_adversary.zero_grad(set_to_none=True)
        with torch.no_grad():
            refreshed_logits = model(images)
        refreshed_adv_logits = model.conditional_sex_logits(
            refreshed_logits.detach(),
            targets,
        )
        adversary_loss = conditional_adv_loss(
            refreshed_adv_logits,
            targets,
            sexes,
            eo_cell_weights,
        )
        adversary_loss.backward()
        torch.nn.utils.clip_grad_norm_(adversary_parameters, max_norm=5.0)
        optimizer_adversary.step()

        total_disease_loss += disease_loss.item() * images.size(0)
        total_adv_loss += adversary_loss.item() * images.size(0)
        lambda_values.append(float(lambda_current))
        cosine_values.append(float(cosine.detach().cpu()))
        task_norm_values.append(float(task_norm.detach().cpu()))
        adv_norm_values.append(float(adv_norm.detach().cpu()))
        effective_adv_ratio_values.append(
            float(
                (reverse_scale * adv_norm / (task_norm + 1e-12))
                .detach()
                .cpu()
            )
        )
        conflict_flags.append(conflict)
        schedule_rows.append(
            {
                "batch": int(batch_index),
                "global_step": int(global_step),
                "fine_tune_epoch": int(fine_tune_epoch),
                "fine_tune_epoch_position": float(
                    global_step / max(len(loader), 1)
                ),
                "lambda_adv": float(lambda_current),
                "lambda_fraction": float(lambda_fraction),
                "task_adv_gradient_cosine": float(cosine.detach().cpu()),
                "task_gradient_norm": float(task_norm.detach().cpu()),
                "adversary_gradient_norm": float(adv_norm.detach().cpu()),
                "effective_reversed_gradient_ratio": float(
                    (reverse_scale * adv_norm / (task_norm + 1e-12))
                    .detach()
                    .cpu()
                ),
                "projection_active": conflict,
                "layer4_trainable": bool(fine_tune_epoch > HEAD_ONLY_EPOCHS),
            }
        )
        global_step += 1

    n = len(loader.dataset)
    return {
        "disease_loss": float(total_disease_loss / n),
        "adversary_loss": float(total_adv_loss / n),
        "lambda_start": float(lambda_values[0]),
        "lambda_end": float(lambda_values[-1]),
        "lambda_mean": float(np.mean(lambda_values)),
        "mean_task_adv_gradient_cosine": float(np.nanmean(cosine_values)),
        "mean_task_gradient_norm": float(np.mean(task_norm_values)),
        "mean_adversary_gradient_norm": float(np.mean(adv_norm_values)),
        "mean_effective_reversed_gradient_ratio": float(
            np.mean(effective_adv_ratio_values)
        ),
        "conflict_fraction": float(np.mean(conflict_flags)),
        "global_step": int(global_step),
        "schedule_rows": schedule_rows,
    }

## Validation metrics and static reference

In [6]:
def evaluate_dynamic_eo_model(
    model,
    loader,
    disease_criterion,
    eo_cell_weights,
    device,
    labels,
):
    model.eval()
    total_disease_loss = 0.0
    total_adv_loss = 0.0
    prob_batches = []
    target_batches = []
    sex_batches = []
    patient_id_batches = []
    image_index_batches = []
    conditional_adv_prob_batches = []

    with torch.no_grad():
        for images, targets, sexes, patient_ids, image_indices in loader:
            images = images.to(device)
            targets = targets.to(device)
            sexes_device = sexes.to(device)

            disease_logits = model(images)
            adv_logits = model.conditional_sex_logits(disease_logits, targets)
            disease_loss = disease_criterion(disease_logits, targets)
            adv_loss = conditional_adv_loss(
                adv_logits,
                targets,
                sexes_device,
                eo_cell_weights,
            )

            total_disease_loss += disease_loss.item() * images.size(0)
            total_adv_loss += adv_loss.item() * images.size(0)
            prob_batches.append(torch.sigmoid(disease_logits).cpu().numpy())
            target_batches.append(targets.cpu().numpy())
            sex_batches.append(sexes.numpy())
            patient_id_batches.append(list(patient_ids))
            image_index_batches.append(list(image_indices))
            conditional_adv_prob_batches.append(
                torch.sigmoid(adv_logits).cpu().numpy()
            )

    probs = np.concatenate(prob_batches, axis=0)
    targets_np = np.concatenate(target_batches, axis=0)
    sexes_np = np.concatenate(sex_batches, axis=0).astype(int)
    conditional_adv_probs = np.concatenate(
        conditional_adv_prob_batches,
        axis=0,
    )

    label_metrics = per_label_ranking_metrics(targets_np, probs, labels)
    per_label_adv_aurocs = [
        safe_auroc(sexes_np, conditional_adv_probs[:, label_index])
        for label_index in range(len(labels))
    ]
    mean_conditional_adv_prob = conditional_adv_probs.mean(axis=1)
    n = len(loader.dataset)

    return {
        "probs": probs,
        "targets": targets_np,
        "sexes": sexes_np,
        "patient_ids": np.asarray(
            [item for batch in patient_id_batches for item in batch],
            dtype=str,
        ),
        "image_indices": np.asarray(
            [item for batch in image_index_batches for item in batch],
            dtype=str,
        ),
        "sex_probs": mean_conditional_adv_prob,
        "conditional_adversary_mean_label_auroc": float(
            np.nanmean(per_label_adv_aurocs)
        ),
        "conditional_adversary_sex_auroc": safe_auroc(
            sexes_np,
            mean_conditional_adv_prob,
        ),
        "disease_loss": float(total_disease_loss / n),
        "conditional_adv_loss": float(total_adv_loss / n),
        "disease_macro_auroc": float(np.nanmean(label_metrics["AUROC"])),
        "disease_macro_auprc": float(np.nanmean(label_metrics["AUPRC"])),
        "label_metrics": label_metrics,
    }


def validation_fairness_from_result(result, labels):
    thresholds_df = select_per_label_thresholds(
        probs=result["probs"],
        targets=result["targets"],
        labels=labels,
        metric=THRESHOLD_METRIC,
    )
    overall_metrics, subgroup_metrics = compute_test_metrics_by_group(
        probs=result["probs"],
        targets=result["targets"],
        sexes=result["sexes"],
        labels=labels,
        thresholds_df=thresholds_df,
    )
    fairness_by_label = build_label_fairness_summary(
        overall_metrics=overall_metrics,
        subgroup_metrics=subgroup_metrics,
        model_name="validation_dynamic_eo",
    )
    return thresholds_df, summarise_overall_fairness(fairness_by_label)


def find_static_validation_eo(seed: int, labels):
    result_dir = Path("results")
    candidate_archives = list(
        result_dir.glob(
            f"r50_static_grl_seed{seed}_validation_predictions.npz"
        )
    )
    candidate_archives += [
        path
        for path in result_dir.glob(
            f"*static*seed{seed}*validation_predictions.npz"
        )
        if "static_matched" not in path.name
    ]
    candidate_archives = list(dict.fromkeys(candidate_archives))

    for archive_path in candidate_archives:
        prefix = str(archive_path).replace(
            "_validation_predictions.npz",
            "",
        )
        threshold_path = Path(f"{prefix}_thresholds.csv")
        if not threshold_path.exists():
            continue

        archive = np.load(archive_path, allow_pickle=False)
        archive_labels = [
            str(value) for value in archive["labels"].tolist()
        ]
        if archive_labels != list(labels):
            continue

        thresholds_df = pd.read_csv(threshold_path)
        overall_metrics, subgroup_metrics = compute_test_metrics_by_group(
            probs=archive["probs"],
            targets=archive["targets"],
            sexes=archive["sexes"].astype(int),
            labels=labels,
            thresholds_df=thresholds_df,
        )
        fairness = summarise_overall_fairness(
            build_label_fairness_summary(
                overall_metrics=overall_metrics,
                subgroup_metrics=subgroup_metrics,
                model_name="static_validation_reference",
            )
        )
        return (
            float(fairness["mean_Equalized_Odds_gap"]),
            str(archive_path),
        )

    return None, None


static_reference_rows = []
for seed in FINAL_SEEDS:
    static_eo, static_path = find_static_validation_eo(seed, selected_labels)
    if static_eo is None:
        raise FileNotFoundError(
            f"Could not find the completed notebook-04 static validation "
            f"archive and thresholds for seed {seed}."
        )
    static_reference_rows.append(
        {
            "seed": seed,
            "static_validation_mean_EO_gap": static_eo,
            "static_validation_archive": static_path,
        }
    )

static_reference_df = pd.DataFrame(static_reference_rows)
STATIC_VALIDATION_EO_MEAN = float(
    static_reference_df["static_validation_mean_EO_gap"].mean()
)
SELECTION_TARGET_EO = float(
    STATIC_VALIDATION_EO_MEAN + STATIC_EO_TOLERANCE
)
print("Static validation EO references:")
display(static_reference_df)
print("Mean static validation EO:", STATIC_VALIDATION_EO_MEAN)
print("Dynamic selection target:", SELECTION_TARGET_EO)

Static validation EO references:


,seed,static_validation_mean_EO_gap,static_validation_archive
0,42,0.085539,results/r50_static_grl_seed42_validation_predi...
1,123,0.073940,results/r50_static_grl_seed123_validation_pred...
2,2026,0.052390,results/r50_static_grl_seed2026_validation_pre...


Mean static validation EO: 0.07062308707454705
Dynamic selection target: 0.07262308707454705


## Train all three seeds

In [7]:
def build_run_config(seed: int, baseline_checkpoint_path: Path) -> Dict:
    return {
        "project_scope": "multi_label_chest_xray_classification",
        "model_variant": EXPERIMENT_ID,
        "schedule_mode": SCHEDULE_MODE,
        "backbone": "ImageNet-pretrained ResNet-50 loaded from selected baseline checkpoint",
        "baseline_checkpoint": str(baseline_checkpoint_path),
        "selected_labels": list(selected_labels),
        "split_protocol": "fixed patient-level train/validation/test split",
        "seed": int(seed),
        "batch_size": int(BATCH_SIZE),
        "max_fine_tune_epochs": int(MAX_FINE_TUNE_EPOCHS),
        "adversary_warmup_epochs": int(ADVERSARY_WARMUP_EPOCHS),
        "lambda_max": float(LAMBDA_MAX),
        "ramp_end_epoch": float(RAMP_END_EPOCH),
        "head_only_epochs": int(HEAD_ONLY_EPOCHS),
        "optimizer": {
            "name": "AdamW",
            "layer4_lr": float(LAYER4_LR),
            "disease_head_lr": float(DISEASE_HEAD_LR),
            "adversary_lr": float(ADVERSARY_LR),
            "weight_decay": float(WEIGHT_DECAY),
        },
        "batch_norm_policy": "all ResNet BatchNorm running statistics and affine parameters frozen",
        "eo_adversary": (
            "per-label adversary receives [sigmoid(disease logit), true label]; "
            "(true-label, sex) cells inverse-frequency weighted"
        ),
        "gradient_combination": (
            "lambda-ramped PCGrad-style projection plus gradient-norm-balanced "
            "reversed adversarial gradient"
        ),
        "checkpoint_selection": (
            "one common epoch across all seeds; maximum aggregate validation "
            "macro AUROC subject to aggregate validation EO <= mean static "
            "validation EO + tolerance"
        ),
        "selection_target_validation_EO_gap": float(SELECTION_TARGET_EO),
        "threshold_protocol": (
            "one F1-maximising threshold per label on each selected seed's "
            "complete validation predictions; fixed unchanged on test"
        ),
        "test_set_hyperparameter_policy": "No test data used for training, thresholding, or checkpoint selection.",
    }


def train_one_seed(seed: int) -> Dict:
    seed_everything(seed)
    run_tag = f"{EXPERIMENT_ID}_seed{seed}"
    output_prefix = Path("results") / run_tag
    baseline_checkpoint_path = (
        Path("checkpoints") / f"r50_baseline_final_seed{seed}_best.pt"
    )
    if not baseline_checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing baseline checkpoint: {baseline_checkpoint_path}"
        )

    train_loader, val_loader, test_loader = build_loaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        label_columns=label_columns,
        seed=seed,
        batch_size=BATCH_SIZE,
    )
    disease_criterion, _, _ = make_disease_criterion(
        train_df,
        label_columns,
        device,
    )
    eo_cell_weights = eo_cell_weights_cpu.to(device)

    model = DynamicEOProjectedResNet50(
        n_labels=len(selected_labels)
    ).to(device)
    load_baseline_checkpoint_into_dynamic_model(
        model,
        baseline_checkpoint_path,
        n_labels=len(selected_labels),
    )

    optimizer_predictor = torch.optim.AdamW(
        [
            {
                "params": model.backbone.layer4.parameters(),
                "lr": LAYER4_LR,
            },
            {
                "params": model.disease_head.parameters(),
                "lr": DISEASE_HEAD_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )
    optimizer_adversary = torch.optim.AdamW(
        model.eo_adversary.parameters(),
        lr=ADVERSARY_LR,
        weight_decay=WEIGHT_DECAY,
    )

    run_config = build_run_config(seed, baseline_checkpoint_path)
    warmup_rows = []
    for warmup_epoch in range(1, ADVERSARY_WARMUP_EPOCHS + 1):
        warmup_loss = train_adversary_warmup_epoch(
            model=model,
            loader=train_loader,
            optimizer_adversary=optimizer_adversary,
            eo_cell_weights=eo_cell_weights,
            device=device,
        )
        warmup_rows.append(
            {
                "seed": seed,
                "warmup_epoch": warmup_epoch,
                "conditional_adv_loss": warmup_loss,
            }
        )
        print(
            f"Seed {seed} | adversary warm-up "
            f"{warmup_epoch}/{ADVERSARY_WARMUP_EPOCHS} | "
            f"loss={warmup_loss:.4f}"
        )

    history_rows = []
    schedule_rows = []
    checkpoint_paths = {}
    global_step = 0

    print("\n" + "=" * 78)
    print("Starting", run_tag)
    print("Baseline checkpoint:", baseline_checkpoint_path)

    for epoch in range(1, MAX_FINE_TUNE_EPOCHS + 1):
        train_result = train_gradnorm_projected_epoch(
            model=model,
            loader=train_loader,
            optimizer_predictor=optimizer_predictor,
            optimizer_adversary=optimizer_adversary,
            disease_criterion=disease_criterion,
            eo_cell_weights=eo_cell_weights,
            device=device,
            fine_tune_epoch=epoch,
            global_step=global_step,
        )
        global_step = train_result["global_step"]

        for row in train_result["schedule_rows"]:
            row.update(
                {
                    "seed": seed,
                    "model_variant": EXPERIMENT_ID,
                }
            )
        schedule_rows.extend(train_result["schedule_rows"])

        val_result = evaluate_dynamic_eo_model(
            model,
            val_loader,
            disease_criterion,
            eo_cell_weights,
            device,
            selected_labels,
        )
        _, val_fairness = validation_fairness_from_result(
            val_result,
            selected_labels,
        )

        checkpoint_path = (
            Path("checkpoints")
            / f"{run_tag}_epoch{epoch:02d}.pt"
        )
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": int(epoch),
                "run_config": run_config,
            },
            checkpoint_path,
        )
        checkpoint_paths[epoch] = str(checkpoint_path)

        record = {
            "model_variant": EXPERIMENT_ID,
            "seed": seed,
            "fine_tune_epoch": epoch,
            "checkpoint_path": str(checkpoint_path),
            "lambda_start": train_result["lambda_start"],
            "lambda_end": train_result["lambda_end"],
            "lambda_mean": train_result["lambda_mean"],
            "layer4_trainable": bool(epoch > HEAD_ONLY_EPOCHS),
            "train_disease_loss": train_result["disease_loss"],
            "train_conditional_adv_loss": train_result["adversary_loss"],
            "mean_task_adv_gradient_cosine": train_result[
                "mean_task_adv_gradient_cosine"
            ],
            "mean_task_gradient_norm": train_result[
                "mean_task_gradient_norm"
            ],
            "mean_adversary_gradient_norm": train_result[
                "mean_adversary_gradient_norm"
            ],
            "mean_effective_reversed_gradient_ratio": train_result[
                "mean_effective_reversed_gradient_ratio"
            ],
            "conflict_fraction": train_result["conflict_fraction"],
            "val_disease_loss": val_result["disease_loss"],
            "val_conditional_adv_loss": val_result[
                "conditional_adv_loss"
            ],
            "val_macro_auroc": val_result["disease_macro_auroc"],
            "val_macro_auprc": val_result["disease_macro_auprc"],
            "val_conditional_adversary_mean_label_auroc": val_result[
                "conditional_adversary_mean_label_auroc"
            ],
            **{
                f"val_{key}": float(value)
                for key, value in val_fairness.items()
            },
        }
        history_rows.append(record)

        print(
            f"Seed {seed} | epoch {epoch}/{MAX_FINE_TUNE_EPOCHS} | "
            f"lambda={train_result['lambda_start']:.4f}"
            f"->{train_result['lambda_end']:.4f} | "
            f"val AUROC={val_result['disease_macro_auroc']:.4f} | "
            f"val AUPRC={val_result['disease_macro_auprc']:.4f} | "
            f"val EO={val_fairness['mean_Equalized_Odds_gap']:.4f} | "
            f"adv/task={train_result['mean_effective_reversed_gradient_ratio']:.4f}"
        )

    history_df = pd.DataFrame(history_rows)
    schedule_df = pd.DataFrame(schedule_rows)
    warmup_df = pd.DataFrame(warmup_rows)

    warmup_df.to_csv(
        f"{output_prefix}_adversary_warmup.csv",
        index=False,
    )
    history_df.to_csv(
        f"{output_prefix}_training_history_all_epochs.csv",
        index=False,
    )
    schedule_df.to_csv(
        f"{output_prefix}_lambda_and_gradient_schedule_stepwise.csv",
        index=False,
    )
    with open(
        f"{output_prefix}_preselection_run_config.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(run_config, handle, indent=2)

    return {
        "seed": seed,
        "output_prefix": str(output_prefix),
        "history": history_df,
        "schedule": schedule_df,
        "checkpoint_paths": checkpoint_paths,
        "run_config": run_config,
    }


training_runs = [train_one_seed(seed) for seed in FINAL_SEEDS]

Seed 42 | adversary warm-up 1/2 | loss=0.6928
Seed 42 | adversary warm-up 2/2 | loss=0.6923

Starting r50_dynamic_eo_gradnorm_common_epoch_v2_seed42
Baseline checkpoint: checkpoints/r50_baseline_final_seed42_best.pt
Seed 42 | epoch 1/12 | lambda=0.0000->0.0250 | val AUROC=0.8100 | val AUPRC=0.2993 | val EO=0.0804 | adv/task=0.0086
Seed 42 | epoch 2/12 | lambda=0.0250->0.0750 | val AUROC=0.8099 | val AUPRC=0.3013 | val EO=0.0790 | adv/task=0.0500
Seed 42 | epoch 3/12 | lambda=0.0750->0.1000 | val AUROC=0.8094 | val AUPRC=0.3011 | val EO=0.0787 | adv/task=0.0913
Seed 42 | epoch 4/12 | lambda=0.1000->0.1000 | val AUROC=0.8094 | val AUPRC=0.3007 | val EO=0.0807 | adv/task=0.1000
Seed 42 | epoch 5/12 | lambda=0.1000->0.1000 | val AUROC=0.8088 | val AUPRC=0.3008 | val EO=0.0694 | adv/task=0.1000
Seed 42 | epoch 6/12 | lambda=0.1000->0.1000 | val AUROC=0.8090 | val AUPRC=0.3006 | val EO=0.0791 | adv/task=0.1000
Seed 42 | epoch 7/12 | lambda=0.1000->0.1000 | val AUROC=0.8071 | val AUPRC=0.2994

## Aggregate validation checkpoint selection

In [8]:
history_all = pd.concat(
    [run["history"] for run in training_runs],
    ignore_index=True,
)
schedule_all = pd.concat(
    [run["schedule"] for run in training_runs],
    ignore_index=True,
)

epoch_summary = (
    history_all.groupby("fine_tune_epoch", as_index=False)
    .agg(
        val_macro_auroc_mean=("val_macro_auroc", "mean"),
        val_macro_auroc_std=("val_macro_auroc", "std"),
        val_macro_auprc_mean=("val_macro_auprc", "mean"),
        val_macro_auprc_std=("val_macro_auprc", "std"),
        val_mean_EO_gap_mean=(
            "val_mean_Equalized_Odds_gap",
            "mean",
        ),
        val_mean_EO_gap_std=(
            "val_mean_Equalized_Odds_gap",
            "std",
        ),
        val_worst_group_recall_mean=(
            "val_mean_Worst_group_Recall",
            "mean",
        ),
        lambda_mean=("lambda_mean", "mean"),
        conflict_fraction_mean=("conflict_fraction", "mean"),
    )
)
epoch_summary["selection_target_EO"] = SELECTION_TARGET_EO
epoch_summary["eligible"] = (
    epoch_summary["val_mean_EO_gap_mean"] <= SELECTION_TARGET_EO
)

eligible_epochs = epoch_summary[epoch_summary["eligible"]].copy()
if not eligible_epochs.empty:
    selected_row = eligible_epochs.sort_values(
        by=[
            "val_macro_auroc_mean",
            "val_macro_auprc_mean",
            "val_worst_group_recall_mean",
            "val_mean_EO_gap_mean",
        ],
        ascending=[False, False, False, True],
    ).iloc[0]
    selection_status = "aggregate_validation_EO_constraint_met"
else:
    selected_row = epoch_summary.sort_values(
        by=[
            "val_mean_EO_gap_mean",
            "val_macro_auroc_mean",
            "val_macro_auprc_mean",
        ],
        ascending=[True, False, False],
    ).iloc[0]
    selection_status = "fallback_lowest_aggregate_validation_EO"

SELECTED_COMMON_EPOCH = int(selected_row["fine_tune_epoch"])

summary_prefix = Path("results") / EXPERIMENT_ID
history_all.to_csv(
    f"{summary_prefix}_all_seed_training_history.csv",
    index=False,
)
schedule_all.to_csv(
    f"{summary_prefix}_all_seed_stepwise_schedule.csv",
    index=False,
)
epoch_summary.to_csv(
    f"{summary_prefix}_aggregate_validation_epoch_selection.csv",
    index=False,
)
static_reference_df.to_csv(
    f"{summary_prefix}_static_validation_references.csv",
    index=False,
)
eo_cell_weight_table.to_csv(
    f"{summary_prefix}_eo_cell_weights.csv",
    index=False,
)

print("\nAggregate validation epoch selection:")
display(epoch_summary)
print("Selection status:", selection_status)
print("Selected common epoch:", SELECTED_COMMON_EPOCH)


Aggregate validation epoch selection:


,fine_tune_epoch,val_macro_auroc_mean,val_macro_auroc_std,val_macro_auprc_mean,val_macro_auprc_std,val_mean_EO_gap_mean,val_mean_EO_gap_std,val_worst_group_recall_mean,lambda_mean,conflict_fraction_mean,selection_target_EO,eligible
0,1,0.809639,0.001726,0.299561,0.006229,0.078389,0.003035,0.421603,0.008648,0.505409,0.072623,False
1,2,0.810198,0.000342,0.301423,0.004161,0.074200,0.007300,0.422245,0.049995,0.506018,0.072623,False
2,3,0.810030,0.000944,0.300836,0.004036,0.066190,0.012937,0.431945,0.091347,0.517579,0.072623,True
3,4,0.810073,0.000602,0.300489,0.002408,0.065312,0.014282,0.437626,0.100000,0.490128,0.072623,True
4,5,0.809931,0.001010,0.300628,0.002188,0.064033,0.010667,0.457775,0.100000,0.487221,0.072623,True
5,6,0.810058,0.001129,0.300418,0.002207,0.061247,0.015889,0.449358,0.100000,0.490940,0.072623,True
6,7,0.809176,0.002150,0.299598,0.001897,0.058412,0.012259,0.443494,0.100000,0.491075,0.072623,True
7,8,0.809159,0.001883,0.299077,0.001993,0.058038,0.007063,0.447016,0.100000,0.487965,0.072623,True
8,9,0.808875,0.002209,0.299260,0.001080,0.058086,0.009901,0.423808,0.100000,0.488979,0.072623,True
9,10,0.808783,0.002344,0.298830,0.001371,0.060858,0.003888,0.436902,0.100000,0.494320,0.072623,True


Selection status: aggregate_validation_EO_constraint_met
Selected common epoch: 4


## Final selected-seed evaluation

In [9]:
def evaluate_selected_seed(training_run: Dict) -> Dict:
    seed = int(training_run["seed"])
    seed_everything(seed)

    _, val_loader, test_loader = build_loaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        label_columns=label_columns,
        seed=seed,
        batch_size=BATCH_SIZE,
    )
    disease_criterion, _, _ = make_disease_criterion(
        train_df,
        label_columns,
        device,
    )
    eo_cell_weights = eo_cell_weights_cpu.to(device)

    model = DynamicEOProjectedResNet50(
        n_labels=len(selected_labels)
    ).to(device)
    checkpoint_path = Path(
        training_run["checkpoint_paths"][SELECTED_COMMON_EPOCH]
    )
    checkpoint = safe_torch_load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    val_result = evaluate_dynamic_eo_model(
        model,
        val_loader,
        disease_criterion,
        eo_cell_weights,
        device,
        selected_labels,
    )
    test_result = evaluate_dynamic_eo_model(
        model,
        test_loader,
        disease_criterion,
        eo_cell_weights,
        device,
        selected_labels,
    )

    thresholds_df = select_per_label_thresholds(
        probs=val_result["probs"],
        targets=val_result["targets"],
        labels=selected_labels,
        metric=THRESHOLD_METRIC,
    )
    test_overall_by_label, test_subgroup_metrics = (
        compute_test_metrics_by_group(
            probs=test_result["probs"],
            targets=test_result["targets"],
            sexes=test_result["sexes"],
            labels=selected_labels,
            thresholds_df=thresholds_df,
        )
    )
    label_fairness_summary = build_label_fairness_summary(
        overall_metrics=test_overall_by_label,
        subgroup_metrics=test_subgroup_metrics,
        model_name=EXPERIMENT_ID,
    )
    fairness_means = summarise_overall_fairness(
        label_fairness_summary
    )

    for frame in (
        test_overall_by_label,
        test_subgroup_metrics,
        label_fairness_summary,
    ):
        frame.insert(0, "seed", seed)
        frame.insert(0, "model_variant", EXPERIMENT_ID)

    selected_history = training_run["history"].loc[
        training_run["history"]["fine_tune_epoch"]
        == SELECTED_COMMON_EPOCH
    ].iloc[0]

    overall_result = pd.DataFrame(
        [
            {
                "model_variant": EXPERIMENT_ID,
                "model": EXPERIMENT_ID,
                "seed": seed,
                "n_labels": len(selected_labels),
                "selected_labels": "|".join(selected_labels),
                "schedule_mode": SCHEDULE_MODE,
                "lambda_max": float(LAMBDA_MAX),
                "selected_common_epoch": int(
                    SELECTED_COMMON_EPOCH
                ),
                "selection_status": selection_status,
                "selection_target_validation_EO_gap": float(
                    SELECTION_TARGET_EO
                ),
                "aggregate_selected_validation_EO_gap": float(
                    selected_row["val_mean_EO_gap_mean"]
                ),
                "selected_seed_validation_EO_gap": float(
                    selected_history[
                        "val_mean_Equalized_Odds_gap"
                    ]
                ),
                "selected_seed_val_macro_auroc": float(
                    selected_history["val_macro_auroc"]
                ),
                "selected_seed_val_macro_auprc": float(
                    selected_history["val_macro_auprc"]
                ),
                "test_macro_auroc": float(
                    test_result["disease_macro_auroc"]
                ),
                "test_macro_auprc": float(
                    test_result["disease_macro_auprc"]
                ),
                "test_conditional_adversary_mean_label_auroc": float(
                    test_result[
                        "conditional_adversary_mean_label_auroc"
                    ]
                ),
                **fairness_means,
            }
        ]
    )

    output_prefix = Path(training_run["output_prefix"])
    thresholds_df.to_csv(
        f"{output_prefix}_thresholds.csv",
        index=False,
    )
    test_overall_by_label.to_csv(
        f"{output_prefix}_test_label_metrics.csv",
        index=False,
    )
    test_subgroup_metrics.to_csv(
        f"{output_prefix}_subgroup_metrics.csv",
        index=False,
    )
    label_fairness_summary.to_csv(
        f"{output_prefix}_label_fairness_summary.csv",
        index=False,
    )
    overall_result.to_csv(
        f"{output_prefix}_overall_results.csv",
        index=False,
    )
    save_prediction_archive(
        f"{output_prefix}_validation_predictions.npz",
        val_result,
        selected_labels,
        adversarial=True,
    )
    save_prediction_archive(
        f"{output_prefix}_test_predictions.npz",
        test_result,
        selected_labels,
        adversarial=True,
    )

    final_config = dict(training_run["run_config"])
    final_config.update(
        {
            "selected_common_epoch": int(
                SELECTED_COMMON_EPOCH
            ),
            "selection_status": selection_status,
            "selected_checkpoint": str(checkpoint_path),
            "aggregate_selected_validation_EO_gap": float(
                selected_row["val_mean_EO_gap_mean"]
            ),
            "test_macro_auroc": float(
                test_result["disease_macro_auroc"]
            ),
            "test_macro_auprc": float(
                test_result["disease_macro_auprc"]
            ),
            "test_mean_EO_gap": float(
                fairness_means["mean_Equalized_Odds_gap"]
            ),
        }
    )
    with open(
        f"{output_prefix}_run_config.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(final_config, handle, indent=2)

    print(
        f"Seed {seed} selected epoch {SELECTED_COMMON_EPOCH} | "
        f"test AUROC={test_result['disease_macro_auroc']:.4f} | "
        f"test AUPRC={test_result['disease_macro_auprc']:.4f} | "
        f"test EO={fairness_means['mean_Equalized_Odds_gap']:.4f}"
    )

    return {
        "seed": seed,
        "output_prefix": str(output_prefix),
        "overall_result": overall_result,
        "test_label_metrics": test_overall_by_label,
        "subgroup_metrics": test_subgroup_metrics,
        "label_fairness_summary": label_fairness_summary,
    }


selected_runs = [
    evaluate_selected_seed(run)
    for run in training_runs
]

overall_all = pd.concat(
    [run["overall_result"] for run in selected_runs],
    ignore_index=True,
)
label_metrics_all = pd.concat(
    [run["test_label_metrics"] for run in selected_runs],
    ignore_index=True,
)
subgroup_metrics_all = pd.concat(
    [run["subgroup_metrics"] for run in selected_runs],
    ignore_index=True,
)
label_fairness_all = pd.concat(
    [run["label_fairness_summary"] for run in selected_runs],
    ignore_index=True,
)

overall_all.to_csv(
    f"{summary_prefix}_seed_level_overall_results.csv",
    index=False,
)
label_metrics_all.to_csv(
    f"{summary_prefix}_seed_level_test_label_metrics.csv",
    index=False,
)
subgroup_metrics_all.to_csv(
    f"{summary_prefix}_seed_level_subgroup_metrics.csv",
    index=False,
)
label_fairness_all.to_csv(
    f"{summary_prefix}_seed_level_label_fairness_summary.csv",
    index=False,
)

aggregate_summary = aggregate_multi_seed_table(
    overall_all,
    group_columns=["model_variant", "schedule_mode"],
    metric_columns=[
        "selected_common_epoch",
        "selected_seed_val_macro_auroc",
        "selected_seed_val_macro_auprc",
        "test_macro_auroc",
        "test_macro_auprc",
        "mean_FNR_gap",
        "mean_FPR_gap",
        "mean_Equalized_Odds_gap",
        "mean_Worst_group_Recall",
        "mean_AUROC_gap",
    ],
)
aggregate_summary.to_csv(
    f"{summary_prefix}_aggregate_summary.csv",
    index=False,
)

print("\nSeed-level selected results:")
display(overall_all)
print("\nAggregate selected results:")
display(aggregate_summary)

Seed 42 selected epoch 4 | test AUROC=0.8114 | test AUPRC=0.3108 | test EO=0.0334
Seed 123 selected epoch 4 | test AUROC=0.8096 | test AUPRC=0.3104 | test EO=0.0287
Seed 2026 selected epoch 4 | test AUROC=0.8130 | test AUPRC=0.3130 | test EO=0.0283

Seed-level selected results:


,model_variant,model,seed,n_labels,selected_labels,schedule_mode,lambda_max,selected_common_epoch,selection_status,selection_target_validation_EO_gap,...,selected_seed_val_macro_auroc,selected_seed_val_macro_auprc,test_macro_auroc,test_macro_auprc,test_conditional_adversary_mean_label_auroc,mean_FNR_gap,mean_FPR_gap,mean_Equalized_Odds_gap,mean_Worst_group_Recall,mean_AUROC_gap
0,r50_dynamic_eo_gradnorm_common_epoch_v2,r50_dynamic_eo_gradnorm_common_epoch_v2,42,7,Infiltration|Effusion|Atelectasis|Nodule|Mass|...,dynamic,0.1,4,aggregate_validation_EO_constraint_met,0.072623,...,0.809431,0.300681,0.811434,0.310760,0.511550,0.032478,0.010178,0.033431,0.478330,0.011995
1,r50_dynamic_eo_gradnorm_common_epoch_v2,r50_dynamic_eo_gradnorm_common_epoch_v2,123,7,Infiltration|Effusion|Atelectasis|Nodule|Mass|...,dynamic,0.1,4,aggregate_validation_EO_constraint_met,0.072623,...,0.810625,0.297992,0.809630,0.310408,0.512999,0.026455,0.012229,0.028724,0.460084,0.013005
2,r50_dynamic_eo_gradnorm_common_epoch_v2,r50_dynamic_eo_gradnorm_common_epoch_v2,2026,7,Infiltration|Effusion|Atelectasis|Nodule|Mass|...,dynamic,0.1,4,aggregate_validation_EO_constraint_met,0.072623,...,0.810162,0.302795,0.812952,0.313047,0.502938,0.027452,0.008282,0.028300,0.448273,0.012995



Aggregate selected results:


,model_variant,schedule_mode,selected_common_epoch_mean,selected_seed_val_macro_auroc_mean,selected_seed_val_macro_auprc_mean,test_macro_auroc_mean,test_macro_auprc_mean,mean_FNR_gap_mean,mean_FPR_gap_mean,mean_Equalized_Odds_gap_mean,...,selected_common_epoch_n_seeds,selected_seed_val_macro_auroc_n_seeds,selected_seed_val_macro_auprc_n_seeds,test_macro_auroc_n_seeds,test_macro_auprc_n_seeds,mean_FNR_gap_n_seeds,mean_FPR_gap_n_seeds,mean_Equalized_Odds_gap_n_seeds,mean_Worst_group_Recall_n_seeds,mean_AUROC_gap_n_seeds
0,r50_dynamic_eo_gradnorm_common_epoch_v2,dynamic,4.0,0.810073,0.300489,0.811339,0.311405,0.028795,0.010229,0.030151,...,3,3,3,3,3,3,3,3,3,3


## Optional bootstrap confidence intervals

In [10]:
if RUN_PATIENT_CLUSTER_BOOTSTRAP:
    bootstrap_archives = []
    for run in selected_runs:
        prefix = Path(run["output_prefix"])
        archive = np.load(
            f"{prefix}_test_predictions.npz",
            allow_pickle=False,
        )
        bootstrap_archives.append(
            {
                "seed": run["seed"],
                "probs": archive["probs"],
                "targets": archive["targets"],
                "sexes": archive["sexes"],
                "patient_ids": archive["patient_ids"],
                "image_indices": archive["image_indices"],
                "labels": archive["labels"],
                "thresholds_df": pd.read_csv(
                    f"{prefix}_thresholds.csv"
                ),
            }
        )

    bootstrap_distribution, bootstrap_ci, bootstrap_seed_points = (
        patient_cluster_bootstrap_multiseed(
            bootstrap_archives,
            n_bootstrap=N_BOOTSTRAP,
            random_seed=BOOTSTRAP_RANDOM_SEED,
        )
    )
    bootstrap_distribution.to_csv(
        f"{summary_prefix}_bootstrap_distribution.csv",
        index=False,
    )
    bootstrap_ci.to_csv(
        f"{summary_prefix}_bootstrap_95ci.csv",
        index=False,
    )
    bootstrap_seed_points.to_csv(
        f"{summary_prefix}_bootstrap_seed_point_estimates.csv",
        index=False,
    )
    display(bootstrap_ci)
else:
    print(
        "Bootstrap disabled. Enable it only after confirming all selected "
        "seed outputs."
    )

if DELETE_NONSELECTED_EPOCH_CHECKPOINTS:
    for run in training_runs:
        for epoch, checkpoint_path in run["checkpoint_paths"].items():
            if int(epoch) != SELECTED_COMMON_EPOCH:
                path = Path(checkpoint_path)
                if path.exists():
                    path.unlink()
    print("Deleted non-selected temporary epoch checkpoints.")

Bootstrap disabled. Enable it only after confirming all selected seed outputs.
Deleted non-selected temporary epoch checkpoints.
